# FedSecureAI — Full Combined & Full LI-Medium Federated AML GPU Runner

This dual-pipeline Google Colab notebook executes full-dataset federated learning training on:
1. **IBM AML FULL_COMBINED Dataset**: Complete 36.33-million-row composite (HI-Small 5.08M + LI-Medium 31.25M), completely replacing the historical 500k-row sample.
2. **IBM AML Full LI-Medium Dataset**: Complete 31.25-million-row dataset (~18.75M train, ~6.25M val, ~6.25M test).

### Architectural Guarantees & Safeguards:
- **Strict Full-Data Enforcement**: The runners reject any `row_limit=200000`, subset samples, or legacy 500k caches.
- **Bounded Host Memory (`< 1.0 GB RAM`)**: Disk-backed memory maps (`np.memmap`) stream feature matrices without loading pandas DataFrames, fully preventing Colab Linux OOM `SIGKILL` crashes.
- **Zero-Copy Federated Client Partitioning**: 3 clients (IID stratified or Non-IID account typology isolated) reference shared disk memory maps via row pointers.
- **Persistent Checkpointing & Atomic Resumption**: Dedicated `resume_manifest.json` tracks round-level checkpoints and status, surviving Colab disconnections.
- **12-Experiment Matrix**: GRU, LSTM, BiLSTM $\times$ FedAvg, FedProx ($\mu=0.0001$) $\times$ IID, Non-IID.

## 1. Environment & GPU Hardware Verification

In [ ]:
!nvidia-smi
import psutil
import shutil
import torch

print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU VRAM        : {vram:.2f} GB")
ram = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage("/").free / (1024**3)
print(f"Host System RAM : {ram:.2f} GB")
print(f"Available Disk  : {disk:.2f} GB")
print("=" * 60)

## 2. Mount Google Drive for Persistent Storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Extract Colab Codebase Package

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/FedSecureAI")
LOCAL_ROOT = Path("/content/FedSecureAI")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(str(LOCAL_ROOT))

zip_candidates = [
    DRIVE_ROOT / "FedSecureAI_FULL_COMBINED_AND_LI_MEDIUM_COLAB_READY_v1.zip",
    Path("/content/FedSecureAI_FULL_COMBINED_AND_LI_MEDIUM_COLAB_READY_v1.zip"),
    DRIVE_ROOT / "FedSecureAI_FULL_LI_MEDIUM_COLAB_READY_v1.zip",
    Path("/content/FedSecureAI_FULL_LI_MEDIUM_COLAB_READY_v1.zip"),
]

zip_path = next((p for p in zip_candidates if p.exists()), None)
if zip_path:
    print(f"Extracting {zip_path} into {LOCAL_ROOT}...")
    !unzip -q -o "{zip_path}" -d "{LOCAL_ROOT}"
    print("Codebase extraction completed successfully.")
else:
    print("WARNING: Package ZIP not found! Please upload FedSecureAI_FULL_COMBINED_AND_LI_MEDIUM_COLAB_READY_v1.zip to /content/ or Google Drive.")

## 4. Install Dependencies

In [ ]:
!pip install -q -r requirements_colab.txt
!pip install -q pytest
# Optional: To install NVIDIA RAPIDS cuDF for GPU-accelerated preprocessing, uncomment below:
# !pip install --extra-index-url=https://pypi.nvidia.com "cudf-cu12"

## 5. Automated Pipeline Unit and Integration Test Suite

In [ ]:
# Execute the test suite covering both Full Combined and Full LI-Medium pipelines
!pytest tests/ -v

# ======================================================================
# SECTION A: FULL-DATASET COMBINED (HI-Small + LI-Medium, 36.33M Rows)
# ======================================================================

### A.1. Locate Sources and Build or Verify FULL_COMBINED Dataset
Constructs the full 36.33M-row dataset via streaming 2-way merge in O(1) memory (<50 MB RAM).

In [ ]:
from pathlib import Path

# Locate source datasets
HI_CANDIDATES = [
    DRIVE_ROOT / "data" / "processed" / "ibm_aml" / "HI-Small",
    LOCAL_ROOT / "data" / "processed" / "ibm_aml" / "HI-Small",
]
LI_CANDIDATES = [
    DRIVE_ROOT / "data" / "processed" / "ibm_aml" / "LI-Medium",
    LOCAL_ROOT / "data" / "processed" / "ibm_aml" / "LI-Medium",
]
COMBINED_CANDIDATES = [
    DRIVE_ROOT / "data" / "processed" / "ibm_aml" / "FULL_COMBINED",
    LOCAL_ROOT / "data" / "processed" / "ibm_aml" / "FULL_COMBINED",
]

hi_dir = next((p for p in HI_CANDIDATES if p.exists() and (p / "train" / "transactions.csv").exists()), None)
li_dir = next((p for p in LI_CANDIDATES if p.exists() and (p / "train" / "transactions.csv").exists()), None)
combined_dir = next((p for p in COMBINED_CANDIDATES if p.exists() and (p / "train" / "transactions.csv").exists()), None)

print(f"HI-Small Path      : {hi_dir}")
print(f"LI-Medium Path     : {li_dir}")
print(f"FULL_COMBINED Path : {combined_dir}")

# If FULL_COMBINED is not built yet, build it directly
if combined_dir is None:
    combined_dir = LOCAL_ROOT / "data" / "processed" / "ibm_aml" / "FULL_COMBINED"
    print(f"Building FULL_COMBINED dataset at {combined_dir}...")
    !python backend/scripts/build_full_combined_dataset.py \
        --hi-dir "{hi_dir}" \
        --li-dir "{li_dir}" \
        --out-dir "{combined_dir}"
else:
    print("Verifying existing FULL_COMBINED dataset...")
    !python backend/scripts/build_full_combined_dataset.py \
        --hi-dir "{hi_dir}" \
        --li-dir "{li_dir}" \
        --out-dir "{combined_dir}" \
        --verify-only

### A.2. Comprehensive Hardware & Preflight Safety Audit
Reports RAM, VRAM, available vs required disk space, source row counts, and cache validity before any training is allowed to run.

In [ ]:
import json
import shutil
import psutil
import torch
from pathlib import Path

combined_output_dir = DRIVE_ROOT / "artifacts" / "full_combined_experiment"
combined_output_dir.mkdir(parents=True, exist_ok=True)

# 1. Hardware checks
ram_avail_gb = psutil.virtual_memory().available / (1024**3)
disk_avail_gb = shutil.disk_usage(str(LOCAL_ROOT)).free / (1024**3)
cuda_ok = torch.cuda.is_available()
vram_gb = (torch.cuda.get_device_properties(0).total_memory / (1024**3)) if cuda_ok else 0.0

print("=" * 75)
print("FEDSECUREAI RESOURCE & HARDWARE PREFLIGHT AUDIT")
print("=" * 75)
print(f"Host System RAM Available   : {ram_avail_gb:.2f} GB (Required >= 2.0 GB)")
print(f"CUDA GPU Detected           : {cuda_ok} ({torch.cuda.get_device_name(0) if cuda_ok else 'CPU'})")
print(f"GPU Dedicated VRAM          : {vram_gb:.2f} GB (Batch 4096 needs ~0.2 GB)")
print(f"Available Disk Space        : {disk_avail_gb:.2f} GB (Required >= 20.0 GB)")
print("-" * 75)

# 2. Run runner preflight verification
!python backend/scripts/run_full_combined_training.py \
    --data-root "{combined_dir}" \
    --output-dir "{combined_output_dir}" \
    --device cuda \
    --verify-only

# Safety assertions
assert disk_avail_gb >= 15.0, f"ABORT: Insufficient disk space ({disk_avail_gb:.1f} GB available). Need >= 15.0 GB."
print("-" * 75)
print("PREFLIGHT VERIFICATION STATUS: 100% PASSED — READY FOR FULL COMBINED TRAINING")
print("=" * 75)

### A.3. Launch FULL_COMBINED 12-Experiment Campaign
> **Safety Guard**: Set `CONFIRM_FULL_COMBINED_TRAINING = True` to launch the multi-hour training run. Supports automatic resumption from `resume_manifest.json` if interrupted.

In [ ]:
# Explicit user start trigger to prevent accidental automatic execution
# Set CONFIRM_FULL_COMBINED_TRAINING = True when ready to begin full 36.33M-row training
CONFIRM_FULL_COMBINED_TRAINING = False

if not CONFIRM_FULL_COMBINED_TRAINING:
    print("TRAINING NOT STARTED: Set CONFIRM_FULL_COMBINED_TRAINING = True to launch the campaign.")
else:
    print("LAUNCHING FULL_COMBINED 12-EXPERIMENT CAMPAIGN ON GPU...")
    !python backend/scripts/run_full_combined_training.py \
        --data-root "{combined_dir}" \
        --output-dir "{combined_output_dir}" \
        --batch-size 4096 \
        --rounds 5 \
        --local-epochs 1 \
        --lr 0.001 \
        --fedprox-mu 0.0001 \
        --seed 42 \
        --device cuda

### A.4. Display FULL_COMBINED Experiment Status

In [ ]:
import pandas as pd
from IPython.display import display

status_csv = combined_output_dir / "results" / "experiment_status.csv"
if status_csv.exists():
    df = pd.read_csv(status_csv)
    display(df)
else:
    print("Status CSV not found yet.")

# ======================================================================
# SECTION B: FULL-DATASET LI-MEDIUM (31.25M Rows)
# ======================================================================

### B.1. Preflight Verification for Full LI-Medium

In [ ]:
li_output_dir = DRIVE_ROOT / "artifacts" / "full_li_medium_experiment"
li_output_dir.mkdir(parents=True, exist_ok=True)

!python backend/scripts/run_full_li_medium_training.py \
    --data-root "{li_dir}" \
    --output-dir "{li_output_dir}" \
    --device cuda \
    --verify-only

### B.2. Launch FULL LI-MEDIUM 12-Experiment Campaign
> **Safety Guard**: Set `CONFIRM_FULL_LI_MEDIUM_TRAINING = True` to launch the multi-hour training run.

In [ ]:
# Set CONFIRM_FULL_LI_MEDIUM_TRAINING = True when ready to begin full 31.25M-row training
CONFIRM_FULL_LI_MEDIUM_TRAINING = False

if not CONFIRM_FULL_LI_MEDIUM_TRAINING:
    print("TRAINING NOT STARTED: Set CONFIRM_FULL_LI_MEDIUM_TRAINING = True to launch the campaign.")
else:
    print("LAUNCHING FULL LI-MEDIUM 12-EXPERIMENT CAMPAIGN ON GPU...")
    !python backend/scripts/run_full_li_medium_training.py \
        --data-root "{li_dir}" \
        --output-dir "{li_output_dir}" \
        --batch-size 4096 \
        --rounds 5 \
        --local-epochs 1 \
        --lr 0.001 \
        --fedprox-mu 0.0001 \
        --seed 42 \
        --device cuda

### B.3. Display FULL LI-MEDIUM Experiment Status

In [ ]:
li_status_csv = li_output_dir / "results" / "experiment_status.csv"
if li_status_csv.exists():
    df_li = pd.read_csv(li_status_csv)
    display(df_li)
else:
    print("LI-Medium Status CSV not found yet.")

## 6. Archive and Backup Artifacts to Google Drive

In [ ]:
import datetime
import shutil

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
backup_zip = DRIVE_ROOT / f"FedSecureAI_EXPERIMENT_RESULTS_{timestamp}.zip"

print(f"Archiving experiment outputs to {backup_zip}...")
!zip -r -q "{backup_zip}" artifacts/full_combined_experiment artifacts/full_li_medium_experiment
print(f"Backup complete! Saved to {backup_zip}")